In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch, RunnableSequence
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables.graph_ascii import draw_ascii
from typing import Literal
from pydantic import BaseModel
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
os.environ['OLLAMA_API_KEY'] = apiKey
llm = ChatOllama(model="gpt-oss:120b-cloud")

In [ ]:
# Example 1 - Sequential Chains with pipes
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="Name 5 famous tourist places of {city}"
)
parser = StrOutputParser()
chain = prompt_1 | llm | parser | prompt_2 | llm | parser
response = chain.invoke({"country": "India"})

print(response)


In [ ]:
# Example 2 - Sequential Chains with RunnableSequence
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="Name 5 famous tourist places of {city}"
)
parser = StrOutputParser()
chain_1 = prompt_1 | llm | parser
chain_2 = prompt_2 | llm | parser
sequence = RunnableSequence(chain_1, chain_2)
response = sequence.invoke({"country": "India"})

print(response)


In [ ]:
# Example 3 - Parallel Chains
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="In which continent {country} is located?"
)
parser = StrOutputParser()
#chain = prompt_1 | llm | parser | prompt_2 | llm | parser
parallel_chain = RunnableParallel(
    {
        "response1": prompt_1 | llm | parser,
        "response2": prompt_2 | llm | parser
    }
)
response = parallel_chain.invoke({"country": "India"})
print(response)


In [ ]:
# Example 4 - Parallel + Sequencial Chains
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Write the independence history of {country} in 3 lines"
)

prompt_2 = PromptTemplate(
    input_variables=["country"],
    template="Write notes on tourist places in {country} in 3 lines"
)
parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {
        "history": prompt_1 | llm | parser,
        "notes": prompt_2 | llm | parser
    }
)

prompt_3 = PromptTemplate(
    input_variables=["history", "notes"],
    template="Write summarizing paragraph of a country having history as {history} and tourist places as {notes} in 5 lines"
)
parser = StrOutputParser()
summarizing_chain = prompt_3 | llm | parser

final_chain = parallel_chain | summarizing_chain

response = final_chain.invoke({"country": "India"})
print(response)


In [ ]:
# Example 5 - Conditional chains
parser = StrOutputParser()

# Structured output schema for sentiment
class FeedbackSentiment(BaseModel):
    sentiment: Literal["positive", "negative"]

sentiment_parser =  PydanticOutputParser(pydantic_object=FeedbackSentiment)
sentiment_prompt = PromptTemplate(
    template="Analyze the sentiment of the following customer feedback and classify it as positive or negative.\n\nFeedback: {feedback} and provide feedback in following format: {response_format}",
    input_variables=["feedback"],
    partial_variables={"response_format": sentiment_parser.get_format_instructions()}
)

sentiment_chain = sentiment_prompt | llm | sentiment_parser

# Positive reply chain
positive_prompt = PromptTemplate.from_template(
    "Write a warm, genuine thank-you response to this positive customer feedback:\n\n{feedback}"
)
positive_chain = positive_prompt | llm | parser

# Negative reply chain
negative_prompt = PromptTemplate.from_template(
    "Write a sincere, empathetic apology and offer assistance for this negative customer feedback:\n\n{feedback}"
)
negative_chain = negative_prompt | llm | parser

# Default chain (fallback)
default_chain = RunnableLambda(lambda: "Thank you for reaching out. We will get back to you shortly.")

# Conditional branching
branch = RunnableBranch(
    (lambda x: x["sentiment"].sentiment == "positive", positive_chain),
    (lambda x: x["sentiment"].sentiment == "negative", negative_chain),
    default_chain
)

# Full pipeline
def run_pipeline(feedback_text):
    sentiment_result = sentiment_chain.invoke({"feedback": feedback_text})
    reply = branch.invoke({"sentiment": sentiment_result, "feedback": feedback_text})
    return reply

# Test it
print(run_pipeline("I absolutely loved your product! It changed my life."))
print(run_pipeline("The service was terrible and nothing worked as expected."))



